In [57]:
from dotenv import load_dotenv
load_dotenv()

True

In [58]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [59]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [71]:
import sys
!{sys.executable} -m pip install gitsource

/workspaces/llm-zoomcamp-2026/.venv/bin/python: No module named pip


In [72]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Total documents loaded: {len(documents)}")

Total documents loaded: 72


In [73]:
# Q1 answer
print("Number of lesson pages:", len(documents))

# Peek at a couple of entries
for doc in documents[:3]:
    print(" -", doc['filename'])

Number of lesson pages: 72
 - 01-agentic-rag/lessons/01-intro.md
 - 01-agentic-rag/lessons/02-environment.md
 - 01-agentic-rag/lessons/03-rag.md


In [74]:
import minsearch

# Build index: content = text field, filename = keyword field
index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)
print("Index built.")

Index built.


In [75]:
QUERY = "How does the agentic loop keep calling the model until it stops?"

results = index.search(QUERY, num_results=5)

print("Top results:")
for r in results:
    print(" -", r['filename'])

# Q2 answer
print("\nQ2 answer — filename of first result:", results[0]['filename'])

Top results:
 - 01-agentic-rag/lessons/14-agentic-loop.md
 - 01-agentic-rag/lessons/15-frameworks.md
 - 01-agentic-rag/lessons/13-function-calling.md
 - 01-agentic-rag/lessons/11-agents-intro.md
 - 01-agentic-rag/lessons/16-other-frameworks.md

Q2 answer — filename of first result: 01-agentic-rag/lessons/14-agentic-loop.md


In [76]:
# Download the helper (run once)
import urllib.request
url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py"
urllib.request.urlretrieve(url, "rag_helper.py")
print("rag_helper.py downloaded")

rag_helper.py downloaded


In [78]:
import os
from openai import OpenAI

XAI_API_KEY = os.environ.get("XAI_API_KEY")

client = OpenAI(
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
)

GROK_MODEL = "grok-3-mini"
print("Client ready:", client.base_url)

Client ready: https://api.x.ai/v1/


In [79]:
# ── Custom RAG that adapts RAGBase for our filename/content schema
# and exposes token usage ──────────────────────────────────────────

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
""".strip()

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


class RAG:
    """RAG adapted for filename/content documents using the Grok (xAI) client."""

    def __init__(self, index, llm_client, model=GROK_MODEL,
                 instructions=INSTRUCTIONS, prompt_template=PROMPT_TEMPLATE):
        self.index = index
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.prompt_template = prompt_template

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(doc['content'])
            lines.append("")
        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(question=query, context=context)

    def llm(self, prompt):
        """Returns (answer_text, usage_object)."""
        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.instructions},
                {"role": "user",   "content": prompt},
            ],
        )
        answer = response.choices[0].message.content
        usage  = response.usage
        return answer, usage

    def rag(self, query):
        """Returns (answer, usage)."""
        search_results = self.search(query)
        prompt         = self.build_prompt(query, search_results)
        answer, usage  = self.llm(prompt)
        return answer, usage


rag = RAG(index=index, llm_client=client)
print("RAG ready")

RAG ready


In [89]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

GROK_MODEL = "llama-3.3-70b-versatile"  # ← this is already correct
print("Client ready:", client.base_url)

Client ready: https://api.groq.com/openai/v1/


In [90]:
rag = RAG(index=index, llm_client=client)
print("RAG ready")

RAG ready


In [93]:
import os
from openai import OpenAI

# Client
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)
GROK_MODEL = "llama-3.3-70b-versatile"

# RAG class
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.
Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
""".strip()

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

class RAG:
    def __init__(self, index, llm_client, model=GROK_MODEL,
                 instructions=INSTRUCTIONS, prompt_template=PROMPT_TEMPLATE):
        self.index = index
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.prompt_template = prompt_template

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(doc['content'])
            lines.append("")
        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(question=query, context=context)

    def llm(self, prompt):
        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.instructions},
                {"role": "user",   "content": prompt},
            ],
        )
        return response.choices[0].message.content, response.usage

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        return self.llm(prompt)

# Instantiate
rag = RAG(index=index, llm_client=client)
print("Model:", rag.model)  # should print llama-3.3-70b-versatile
print("Ready!")

Model: llama-3.3-70b-versatile
Ready!


In [94]:
answer_q3, usage_q3 = rag.rag(QUERY)
print(usage_q3.prompt_tokens)

7221


In [95]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

# Q4 answer
print("Number of chunks:", len(chunks))
print("Sample chunk keys:", list(chunks[0].keys()))

Number of chunks: 295
Sample chunk keys: ['start', 'content', 'filename']


In [96]:
# Build a new index on chunks
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(chunks)
print("Chunk index built")

Chunk index built


In [97]:
rag_chunked = RAG(index=chunk_index, llm_client=client)

answer_q5, usage_q5 = rag_chunked.rag(QUERY)

print("Answer:")
print(answer_q5)
print()
print("Usage:", usage_q5)
print(f"\nQ3 prompt tokens : {usage_q3.prompt_tokens}")
print(f"Q5 prompt tokens : {usage_q5.prompt_tokens}")
ratio = usage_q3.prompt_tokens / usage_q5.prompt_tokens
print(f"Ratio (Q3/Q5)    : {ratio:.1f}x fewer tokens with chunking")

Answer:
The agentic loop keeps calling the model until it stops by using a `while True` loop that continues to execute until a certain condition is met. The condition is that the model returns a response without any function calls. This is checked by setting a flag `has_function_calls` to `True` if any function calls are found in the model's response, and `False` otherwise. If `has_function_calls` is `False` after processing the model's response, the loop breaks, indicating that the model has stopped asking for tools and the loop can terminate. 

In the provided code, this is implemented as follows:

```python
while True:
    ...
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            ...
            has_function_calls = True

    it = it + 1
    if has_

In [99]:
import subprocess, sys
subprocess.run(["uv", "pip", "install", "toyaikit"], check=True)

Resolved 41 packages in 674ms
Prepared 7 packages in 286ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 7 packages in 393ms
 + anthropic==0.111.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.66
 + httpcore2==2.4.0
 + httpx2==2.4.0
 + toyaikit==0.0.11
 + truststore==0.10.4


CompletedProcess(args=['uv', 'pip', 'install', 'toyaikit'], returncode=0)

In [106]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

GROK_MODEL = "llama3-groq-70b-8192-tool-use-preview"
print("Client ready, model:", GROK_MODEL)

Client ready, model: llama3-groq-70b-8192-tool-use-preview


In [109]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

GROK_MODEL = "llama-3.1-8b-instant"
print("Client ready, model:", GROK_MODEL)

Client ready, model: llama-3.1-8b-instant


In [110]:
import json

search_call_count = 0

def search(query: str) -> str:
    """Search the course lessons for relevant information."""
    global search_call_count
    search_call_count += 1
    print(f"  [search call #{search_call_count}] query: {query!r}")
    results = chunk_index.search(query, num_results=3)
    lines = []
    for r in results:
        lines.append(f"File: {r['filename']}")
        lines.append(r['content'])
        lines.append("---")
    return "\n".join(lines)

tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the course lesson knowledge base.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

AGENT_INSTRUCTIONS = (
    "You're a course teaching assistant. "
    "Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)
AGENT_QUERY = "How does the agentic loop work, and how is it different from plain RAG?"

search_call_count = 0
messages = [
    {"role": "system", "content": AGENT_INSTRUCTIONS},
    {"role": "user",   "content": AGENT_QUERY},
]

while True:
    response = client.chat.completions.create(
        model=GROK_MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )
    msg = response.choices[0].message
    messages.append(msg)

    if not msg.tool_calls:
        break

    for tool_call in msg.tool_calls:
        args = json.loads(tool_call.function.arguments)
        result = search(args["query"])
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result,
        })

print("\n=== Agent's Final Answer ===")
print(msg.content)
print(f"\nQ6 answer — search tool called {search_call_count} time(s)")

  [search call #1] query: 'agentic loop vs RAG'
  [search call #2] query: 'agentic loop explanation'
  [search call #3] query: 'agentic loop vs plain RAG differences'

=== Agent's Final Answer ===
Based on the search results, the agentic loop is a software framework that enables users to build interactive chatbots using a modular and extensible architecture. It provides a set of tools and techniques for building conversational AI models that can be composed together to create complex chatbots.

The agentic loop is different from plain RAG (Reasoning Augmented Graph) in that it uses a more flexible and dynamic approach to building chatbots. Unlike RAG, which relies on a fixed set of steps and tools, the agentic loop allows users to define their own sets of tools and agents, and to compose them together in a flexible and dynamic way.

This approach enables users to build chatbots that can adapt to changing information and unexpected conditions, and to use a wide range of tools and agents

In [111]:
print("=== HOMEWORK ANSWERS ===")
print(f"Q1  Number of lesson pages  : {len(documents)}")
print(f"Q2  Filename of top result  : {results[0]['filename']}")
print(f"Q3  Input tokens (full docs): {usage_q3.prompt_tokens}")
print(f"Q4  Number of chunks        : {len(chunks)}")
print(f"Q5  Input tokens (chunks)   : {usage_q5.prompt_tokens}  (~{ratio:.0f}x fewer)")
print(f"Q6  Agent search() calls    : {search_call_count}")

=== HOMEWORK ANSWERS ===
Q1  Number of lesson pages  : 72
Q2  Filename of top result  : 01-agentic-rag/lessons/14-agentic-loop.md
Q3  Input tokens (full docs): 7221
Q4  Number of chunks        : 295
Q5  Input tokens (chunks)   : 2348  (~3x fewer)
Q6  Agent search() calls    : 3
